## 시즌 아이디(seasonId) 메타데이터 조회

> `@../docs/10-FCONLINE/14-Meta-Data.md`

### 개요

`GET /static/fconline/meta/seasonid.json` 으로 전체 시즌(클래스) 목록을 수집한다.  
인증 불필요. 총 **141건** 수록.

| 컬럼 | 타입 | 원본 필드 | 변환 규칙 |
|---|---|---|---|
| `ID` | int | `seasonId` | 그대로 |
| `class_ID` | str | `seasonImg` | 파일명에서 `.png` 제거 후 대문자 (e.g. `ICONTM`) |
| `class_name` | str | `className` | `()` 안의 내용 추출 (e.g. `ICON The Moment`) |
| `season_IMG` | str | `seasonImg` | 그대로 |
| `season_big_IMG` | str | `seasonImg` | `new/season/` 경로에 파일명 `{name}_big.png` 형태 (⚠️ 없는 시즌도 있음.) |
| `card_IMG` | str | `seasonImg` | URL 내 `season` → `card` 치환 |

### ERD

```mermaid
erDiagram
    meta_seasonid {
        INT season_ID PK
        STRING class_ID
        STRING class_name
        STRING season_IMG
        STRING season_big_IMG
        STRING card_IMG
    }
```

### 저장 위치

```
data/meta_seasonid.csv
```

In [2]:
import csv
import os
import re

import requests
from dotenv import load_dotenv

load_dotenv()

_BASE_URL = os.getenv("SERVERS")
_SEASONID_URL = f"{_BASE_URL}/static/fconline/meta/seasonid.json"
_BIG_IMG_BASE = "https://ssl.nexon.com/s2/game/fc/online/obt/externalAssets/new/season"

OUTPUT = "../data/meta_seasonid.csv"


def fetch_seasonid_meta() -> list[dict]:
    resp = requests.get(_SEASONID_URL)
    resp.raise_for_status()
    return resp.json()


def parse_seasonid(raw: list[dict]) -> list[dict]:
    result = []
    for item in raw:
        season_img = item["seasonImg"]
        class_id = season_img.split("/")[-1].replace(".png", "").upper()
        m = re.search(r"\(([^)]+)\)\s*$", item["className"])
        class_name = m.group(1) if m else item["className"]
        filename = season_img.split("/")[-1].replace(".png", "")
        result.append({
            "ID": item["seasonId"],
            "class_ID": class_id,
            "class_name": class_name,
            "season_IMG": season_img,
            "season_big_IMG": f"{_BIG_IMG_BASE}/{filename}_big.png",
            "card_IMG": season_img.replace("season", "card")
        })
    return result


def save_csv(records: list[dict], path: str) -> None:
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["ID", "class_ID", "class_name", "season_IMG", "season_big_IMG", "card_IMG"],
        )
        writer.writeheader()
        writer.writerows(records)
    print(f"저장 완료: {path} ({len(records)}건)")


if __name__ == "__main__":
    print("seasonid 메타데이터 수집 중...")
    raw = fetch_seasonid_meta()
    records = parse_seasonid(raw)
    print(f"총 {len(records)}건")
    save_csv(records, OUTPUT)


python-dotenv could not parse statement starting at line 15


seasonid 메타데이터 수집 중...
총 141건
저장 완료: ../data/meta_seasonid.csv (141건)
